# brinkgrad Quickstart Tutorial

**brinkgrad** is an open-source FEniCSx framework for adjoint-based topology
optimisation of coupled Brinkman–convection-diffusion systems in porous media.

This notebook demonstrates:
1. Running a linear gradient optimisation
2. Changing the target profile (one argument)
3. Inspecting results (RMSE, convergence history)
4. Post-processing with robust projection

**Requirements:** Run inside the Docker container:
```bash
docker run --rm -it -v ${PWD}:/root/brinkgrad dolfinx/dolfinx:v0.7.3 bash
pip install -e /root/brinkgrad
jupyter notebook
```

In [ ]:
# Install brinkgrad (if not already installed)
import subprocess
subprocess.run(['pip', 'install', '-e', '..', '--quiet'], check=True)
print('brinkgrad installed')

## 1. Linear gradient target (primary benchmark)

In [ ]:
from brinkgrad import GradientGeneratorOptimizer

# Define optimiser — linear gradient target c*(y) = y/Ly
opt = GradientGeneratorOptimizer(
    Lx=2000e-6, Ly=500e-6,
    nx=20, ny=5,              # coarse mesh for quick demo (use 80x20 for paper results)
    target_expr=lambda x: x[1]/500e-6,
    w_f=1e-3, w_c=50.0,
    V_star=0.5)

# Run 20 iterations (use max_iter=600 or 1400 for paper results)
opt.run(max_iter=20)
print(f'RMSE = {opt.rmse:.4f}')
print(f'Gray-zone fraction = {opt.gray_frac:.3f}')

## 2. Changing the target profile (one argument change)

In [ ]:
import numpy as np
from brinkgrad import GradientGeneratorOptimizer

# Step profile: c*(y) = 1 if y > Ly/2, else 0
Ly = 500e-6
step = lambda x: np.where(x[1] > Ly/2, 1.0, 0.0)

opt_step = GradientGeneratorOptimizer(
    Lx=2000e-6, Ly=Ly, nx=20, ny=5,
    target_expr=step,   # <-- only this line changes
    w_f=1e-3, w_c=50.0, V_star=0.5)
opt_step.run(max_iter=20)
print(f'Step target RMSE = {opt_step.rmse:.4f}')

## 3. Visualise convergence history

In [ ]:
import matplotlib.pyplot as plt

if hasattr(opt, 'J_history') and opt.J_history:
    plt.figure(figsize=(8,3))
    plt.plot(opt.J_history, 'b-', linewidth=1.5)
    plt.xlabel('Iteration')
    plt.ylabel('Objective J')
    plt.title('Convergence history (linear gradient target)')
    plt.tight_layout()
    plt.savefig('convergence_demo.pdf')
    plt.show()
    print('Saved convergence_demo.pdf')
else:
    print('J_history not available — run with more iterations')

## 4. Robust projection (binary design post-processing)

In [ ]:
from brinkgrad.manufacturability import robust_projection

# Apply robust projection to obtain near-binary design
# See examples/robust_projection_demo.py for full workflow
print('robust_projection() is available for binary design post-processing')
print('See examples/robust_projection_demo.py for a complete example')

## 5. Reproduce paper results

To reproduce all paper results (RMSE = 0.058, 1400 iterations on 80×20 mesh):

```bash
# Inside Docker container:
bash run_all.sh
```

Or run individual examples:
```bash
python examples/linear_target.py      # primary benchmark
python examples/step_profile_target.py
python examples/double_peak_target.py
python examples/robust_projection_demo.py
```

All results are archived at Zenodo: https://doi.org/10.5281/zenodo.20538833